# Stage 2 single defect training

## Cell 1 - GPU check

In [ ]:
import torch, sys, os

print(f'Python  : {sys.version[:6]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found. Enable GPU in notebook settings before continuing.')

## Cell 2 - Install dependencies

In [ ]:
import subprocess, sys

packages = [
    "diffusers==0.29.2",
    "transformers==4.40.0",
    "accelerate==0.29.3",
    "peft==0.10.0",
    "wandb>=0.18.0",
    "safetensors",
    "Pillow",
    "tqdm",
    "pyyaml",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade",
    *packages
])

print("Installation complete. Verifying versions...\n")

import importlib
expected = {
    "diffusers":    "0.29",
    "transformers": "4.40",
    "accelerate":   "0.29",
    "peft":         "0.10",
}

all_ok = True
for pkg, prefix in expected.items():
    mod = importlib.import_module(pkg)
    ver = mod.__version__
    ok  = ver.startswith(prefix)
    print(f"  {'ok' if ok else 'X! WRONG'}  {pkg:<20} {ver}")
    if not ok:
        all_ok = False

import numpy as np
import wandb
print(f"  ok  numpy                {np.__version__}")
print(f"  ok  wandb                {wandb.__version__}")

if not all_ok:
    raise RuntimeError("Wrong package versions - restart kernel and re-run")
print("\nAll critical packages correct - safe to continue")

## Cell 3 - NumPy patch

In [ ]:
import numpy as np

if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'complex_'):
    np.complex_ = np.complex128

print(f'NumPy {np.__version__} - patch applied.')

## Cell 4 - Authenticate

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()

hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('HuggingFace : OK')

wandb_token = secrets.get_secret('WANDB_API_KEY')
wandb.login(key=wandb_token)
print('Wandb       : OK')

## Cell 5 - Clone repo and set working directory

In [ ]:
import os
from pathlib import Path

WORK     = '/kaggle/working'
REPO_URL = 'https://github.com/alecanc/Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
REPO_DIR = f'{WORK}/repo'

for d in ['checkpoints/stage2', 'splits', 'generated']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory : {os.getcwd()}')
print(f'Repo contents     : {sorted(os.listdir("."))}')

CONFIG_PATH = f'{REPO_DIR}/defect-synthesis/config.yaml'
print(f'Config path       : {CONFIG_PATH}')

## Cell 6 - Mount MVTec dataset

If dataset is already extracted.  
Verify INPUT_DIR matches the path shown in the Kaggle Data sidebar.  
All three categories must be present for the full training run.

In [ ]:
from pathlib import Path

# ── Verify the path in the Kaggle sidebar > Data panel ──────────────────────
INPUT_DIR = Path('/kaggle/input/datasets/ipythonx/mvtec-ad')
# ─────────────────────────────────────────────────────────────────────────────

MVTEC_ROOT = INPUT_DIR

CATEGORIES = {
    'bottle':    ['broken_large', 'broken_small', 'contamination'],
    'metal_nut': ['bent', 'color', 'flip', 'scratch'],
    'leather':   ['color', 'cut', 'fold', 'glue', 'poke'],
}

print('MVTec dataset verification:')
all_ok = True
for cat, dtypes in CATEGORIES.items():
    cat_dir = MVTEC_ROOT / cat
    if not cat_dir.exists():
        print(f'  X!  {cat} - NOT FOUND at {cat_dir}')
        all_ok = False
        continue
    n_clean = len(list((cat_dir / 'train/good').glob('*.png')))
    print(f'  ok  {cat} - {n_clean} clean train images')
    for dt in dtypes:
        dt_dir = cat_dir / 'test' / dt
        n = len(list(dt_dir.glob('*.png'))) if dt_dir.exists() else 0
        status = 'ok' if n > 0 else 'X! MISSING'
        print(f'       {status}  {dt:<20} {n} images')

if not all_ok:
    raise FileNotFoundError('Some categories are missing. Check INPUT_DIR.')
print('\nAll categories verified.')

## Cell 7 - Patch config.yaml and generate splits

Only overwrites runtime paths, all hyperparameters are trusted from the committed config.

In [ ]:
import yaml, json

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['paths']['mvtec_root']  = str(MVTEC_ROOT)
cfg['paths']['output_root'] = WORK
cfg['paths']['checkpoints'] = f'{WORK}/checkpoints'
cfg['paths']['generated']   = f'{WORK}/generated'
cfg['paths']['splits_dir']  = f'{WORK}/splits'
cfg['paths']['logs']        = f'{WORK}/logs'

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('config.yaml - runtime paths applied.')
print(f"  model_id        : {cfg['model_id']}")
print(f"  mvtec_root      : {cfg['paths']['mvtec_root']}")
print(f"  max_train_steps : {cfg['stage2']['max_train_steps']}")
print(f"  n_per_type      : {cfg['splits']['stage2_n_per_type']}")
print()

# Generate splits for all three categories
!python data/splits.py --config {CONFIG_PATH}

# Verify all three split files and spot-check paths
print()
splits_dir = Path(f'{WORK}/splits')
for cat in ['bottle', 'metal_nut', 'leather']:
    split_file = splits_dir / f'{cat}_split.json'
    if not split_file.exists():
        print(f'  X!  {cat}_split.json NOT FOUND')
        continue
    with open(split_file) as f:
        split = json.load(f)
    # Spot-check first path exists on disk
    first_dtype = list(split['stage2'].keys())[0]
    first_path  = Path(split['stage2'][first_dtype][0])
    ok = 'ok' if first_path.exists() else 'X! PATH BROKEN'
    print(f'  {ok}  {cat} split - {sum(len(v) for v in split["stage2"].values())} defect images total')
    for dtype, imgs in split['stage2'].items():
        print(f'       {dtype:<20} {len(imgs)} train | {len(split["eval"].get(dtype, []))} eval')

## Custom train one defect only


In [ ]:
# -- SINGLE DEFECT RETRAIN: new parameters ----------------------------
RETRAIN_CATEGORY    = 'metal_nut'
RETRAIN_DEFECT_TYPE = 'bent'
RETRAIN_LORA_RANK   = 8
RETRAIN_LORA_ALPHA  = 32            # scaling alpha/rank = 4
RETRAIN_MAX_STEPS   = 400 
RETRAIN_LR          = 5e-5
# ---------------------------------------------------------------------

import yaml

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['stage2']['lora_rank']       = RETRAIN_LORA_RANK
cfg['stage2']['lora_alpha']      = RETRAIN_LORA_ALPHA
cfg['stage2']['max_train_steps'] = RETRAIN_MAX_STEPS
cfg['stage2']['learning_rate']   = RETRAIN_LR

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Config overridden for {RETRAIN_CATEGORY}/{RETRAIN_DEFECT_TYPE}:')
print(f'  lora_rank       : {RETRAIN_LORA_RANK}')
print(f'  lora_alpha      : {RETRAIN_LORA_ALPHA}')
print(f'  max_train_steps : {RETRAIN_MAX_STEPS}')
print(f'  learning_rate   : {RETRAIN_LR}')

In [ ]:
# FALLBACK, leave commented. Enable only if single-stage grids show low
# diversity or overfitting to the defect scenes. Mixes n_clean clean product
# images (identity prompt) into training to reinforce object identity.
# import yaml
# with open(CONFIG_PATH) as f:
#     cfg = yaml.safe_load(f)
# cfg['stage2']['include_clean'] = True
# cfg['stage2']['n_clean'] = 8
# with open(CONFIG_PATH, 'w') as f:
#     yaml.dump(cfg, f, default_flow_style=False)
# print('Identity reinforcement enabled: include_clean=True, n_clean=8')

In [ ]:
import shutil
from pathlib import Path

print(f"\n{'='*60}")
print(f'  Retraining: {RETRAIN_CATEGORY} / {RETRAIN_DEFECT_TYPE}')
print(f"{'='*60}")
!python -u train/train_stage2.py \
    --config {CONFIG_PATH} \
    --category {RETRAIN_CATEGORY} \
    --defect_type {RETRAIN_DEFECT_TYPE}

source   = f'{WORK}/checkpoints/stage2/{RETRAIN_CATEGORY}/{RETRAIN_DEFECT_TYPE}'
zip_base = f'{WORK}/stage2_{RETRAIN_CATEGORY}_{RETRAIN_DEFECT_TYPE}_singlestage'
shutil.make_archive(zip_base, 'zip', source)
zip_path = Path(f'{zip_base}.zip')
print(f'\n  > Zipped: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')

## Verify

In [ ]:
from pathlib import Path

final = Path(f'{WORK}/checkpoints/stage2/{RETRAIN_CATEGORY}/{RETRAIN_DEFECT_TYPE}/final')
st    = next(final.glob('*.safetensors'), None) if final.exists() else None

if st is None:
    print(f'  ✗  {RETRAIN_CATEGORY}/{RETRAIN_DEFECT_TYPE}  MISSING')
else:
    size = st.stat().st_size / 1e6
    ok   = 1.0 < size < 50.0
    print(f'  {"ok" if ok else "x! WRONG SIZE"}  {RETRAIN_CATEGORY}/{RETRAIN_DEFECT_TYPE}  {size:.1f} MB')

## Generate Grid

In [ ]:
import torch, yaml
from diffusers import StableDiffusionPipeline
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display
from peft import PeftModel

N_IMAGES = 8
N_COLS   = 4

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

model_id    = cfg['model_id']
token_D     = cfg['token_D']
token_V     = cfg['token_V']
prompt      = f'a photo of a {token_V} {RETRAIN_CATEGORY} with a {token_D} defect'
adapter_dir = Path(f'{WORK}/checkpoints/stage2/{RETRAIN_CATEGORY}/{RETRAIN_DEFECT_TYPE}/final')

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')
pipe.set_progress_bar_config(disable=True)
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_dir))
pipe.unet = pipe.unet.to(dtype=torch.float16)
pipe.vae  = pipe.vae.to(dtype=torch.float32)

generator = torch.Generator('cuda').manual_seed(cfg['seed'])
images    = pipe(
    prompt,
    num_images_per_prompt=N_IMAGES,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images

w, h   = images[0].size
n_rows = (N_IMAGES + N_COLS - 1) // N_COLS
grid   = PILImage.new('RGB', (w * N_COLS, h * n_rows))
for i, img in enumerate(images):
    grid.paste(img, ((i % N_COLS) * w, (i // N_COLS) * h))

out_path = Path(f'{WORK}/generated/{RETRAIN_CATEGORY}_{RETRAIN_DEFECT_TYPE}_singlestage.png')
grid.save(out_path)
print(f'Saved: {out_path.name}')
display(grid)

del pipe
torch.cuda.empty_cache()